<a href="https://colab.research.google.com/github/Aryanjha24/CandidateElimination/blob/main/CandidateElimination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#For a given set of training data examples stored in a .CSV file,
#implement and demonstrate the Candidate-Elimination algorithm to output a description
#of the set of all hypotheses consistent with the training examples.
from google.colab import files
import pandas as pd

# ---- Upload your CSV (last column = target class Yes/No) ----
uploaded = files.upload()
csv_path = list(uploaded.keys())[0]

PHI = "0"   # most-specific ("null") constraint symbol
ANY = "?"        # most-general ("don't care") constraint symbol

def load_training_data(csv_path):
    df = pd.read_csv(csv_path)
    attr_names = list(df.columns[:-1])
    target_col = df.columns[-1]
    positive_labels = {"yes", "y", "1", "true", "positive"}

    examples = []
    for _, row in df.iterrows():
        instance = tuple(str(row[a]) for a in attr_names)
        label = str(row[target_col]).strip().lower()
        examples.append((instance, label in positive_labels))

    domains = [set(df[a].astype(str).unique()) for a in attr_names]
    return examples, attr_names, domains


def more_general_or_equal(h1, h2):
    """h1 is more general than or equal to h2."""
    for a1, a2 in zip(h1, h2):
        if a1 == ANY:
            continue
        if a2 == PHI:
            continue
        if a1 == PHI:
            return False
        if a1 != a2:
            return False
    return True


def consistent(instance, hypothesis):
    """An instance is consistent with (covered by) a hypothesis."""
    return more_general_or_equal(hypothesis, instance)


def minimal_generalizations(s, instance):
    """Minimal generalization(s) of s that cover the positive instance."""
    new_s = []
    for s_i, x_i in zip(s, instance):
        if s_i == PHI:
            new_s.append(x_i)
        elif s_i == x_i:
            new_s.append(s_i)
        else:
            new_s.append(ANY)
    return tuple(new_s)


def minimal_specializations(g, domains, instance):
    """Minimal specialization(s) of g that exclude the negative instance."""
    result = []
    for i in range(len(g)):
        if g[i] == ANY:
            for value in domains[i]:
                if value != instance[i]:
                    new_g = list(g)
                    new_g[i] = value
                    result.append(tuple(new_g))
    return result


def drop_more_general(S):
    """Remove from S any hypothesis more general than another in S."""
    keep = [s for s in S if not any(s != s2 and more_general_or_equal(s, s2) for s2 in S)]
    return list(dict.fromkeys(keep))


def drop_less_general(G):
    """Remove from G any hypothesis less general than another in G."""
    keep = [g for g in G if not any(g != g2 and more_general_or_equal(g2, g) for g2 in G)]
    return list(dict.fromkeys(keep))


def candidate_elimination(examples, domains, verbose=True):
    n = len(examples[0][0])

    # --- "Initialize G to the set of maximally general hypotheses in H"
    G = [tuple([ANY] * n)]
    # --- "Initialize S to the set of maximally specific hypotheses in H"
    S = [tuple([PHI] * n)]

    if verbose:
        print("Initial hypotheses (before processing any training example):")
        print(f"  S0 = {S}")
        print(f"  G0 = {G}")

    # --- "For each training example d, do"
    for step, (d, is_positive) in enumerate(examples, start=1):

        # --- "If d is a positive example"
        if is_positive:
            # "Remove from G any hypothesis inconsistent with d"
            G = [g for g in G if consistent(d, g)]

            # "For each hypothesis s in S that is not consistent with d"
            new_S = []
            for s in S:
                if consistent(d, s):
                    new_S.append(s)
                else:
                    # "Remove s from S" (implicit: not re-added below)
                    # "Add to S all minimal generalizations h of s such that
                    #  h is consistent with d, and some member of G
                    #  is more general than h"
                    h = minimal_generalizations(s, d)
                    if any(more_general_or_equal(g, h) for g in G):
                        new_S.append(h)
            # "Remove from S any hypothesis that is more general
            #  than another hypothesis in S"
            S = drop_more_general(new_S)

        # --- "If d is a negative example"
        else:
            # "Remove from S any hypothesis inconsistent with d"
            S = [s for s in S if not consistent(d, s)]

            # "For each hypothesis g in G that is not consistent with d"
            new_G = []
            for g in G:
                if not consistent(d, g):
                    new_G.append(g)
                else:
                    # "Remove g from G" (implicit)
                    # "Add to G all minimal specializations h of g such that
                    #  h is consistent with d, and some member of S
                    #  is more specific than h"
                    for h in minimal_specializations(g, domains, d):

                        if any(more_general_or_equal(h, s) for s in S):
                            new_G.append(h)
            # "Remove from G any hypothesis that is less general
            #  than another hypothesis in G"
            G = drop_less_general(new_G)

        if verbose:
            print(f"\nExample {step}: {d} -> {'Yes' if is_positive else 'No'}")
            print(f"  S{step} = {S}")
            print(f"  G{step} = {G}")

    return S, G


# ---------------- Run it ----------------
examples, attr_names, domains = load_training_data(csv_path)
print("Attributes:", attr_names)

S_final, G_final = candidate_elimination(examples, domains, verbose=True)

print("\n" + "=" * 60)
print("FINAL VERSION SPACE")
print("=" * 60)
print("S (most specific):", ["<" + ", ".join(h) + ">" for h in S_final])
print("G (most general): ", ["<" + ", ".join(h) + ">" for h in G_final])

Saving enjoysport.csv to enjoysport.csv
Attributes: ['sky', 'airtemp', 'humidity', 'wind', 'water', 'forcast']
Initial hypotheses (before processing any training example):
  S0 = [('0', '0', '0', '0', '0', '0')]
  G0 = [('?', '?', '?', '?', '?', '?')]

Example 1: ('sunny', 'warm', 'normal', 'strong', 'warm', 'same') -> Yes
  S1 = [('sunny', 'warm', 'normal', 'strong', 'warm', 'same')]
  G1 = [('?', '?', '?', '?', '?', '?')]

Example 2: ('sunny', 'warm', 'high', 'strong', 'warm', 'same') -> Yes
  S2 = [('sunny', 'warm', '?', 'strong', 'warm', 'same')]
  G2 = [('?', '?', '?', '?', '?', '?')]

Example 3: ('rainy', 'cold', 'high', 'strong', 'warm', 'change') -> No
  S3 = [('sunny', 'warm', '?', 'strong', 'warm', 'same')]
  G3 = [('sunny', '?', '?', '?', '?', '?'), ('?', 'warm', '?', '?', '?', '?'), ('?', '?', '?', '?', '?', 'same')]

Example 4: ('sunny', 'warm', 'high', 'strong', 'cool', 'change') -> Yes
  S4 = [('sunny', 'warm', '?', 'strong', '?', '?')]
  G4 = [('sunny', '?', '?', '?', '